In [0]:
from pyspark.sql.functions import *

In [0]:
orders = spark.read.table("ecom.silver.orders")
order_items = spark.read.table("ecom.silver.orders_items")
order_payments = spark.read.table("ecom.silver.orders_payments")
customers = spark.read.table("ecom.silver.customers")
products = spark.read.table("ecom.silver.products")


####Revenue by product category

In [0]:
gold_category = orders.join(order_items, "order_id", "inner") \
                        .join(products, "product_id","left")\
                            .filter(col("order_status") == "delivered") \
                            .withColumn("product_category_name", coalesce(col("product_category_name"), lit("uncategorized")))\
                            .groupBy("product_category_name") \
                            .agg(
                                count("order_id").alias("total_orders"),
                                round(sum("price"), 2).alias("total_revenue"),
                                round(sum("freight_value"), 2).alias("total_freight")
                            ) \
                            .orderBy(col("total_revenue").desc()) \
                            .withColumn("_gold_time", current_timestamp())

gold_category.write.format("delta") \
                    .mode("overwrite") \
                    .saveAsTable("ecom.gold.revenue_by_category")

#### Revenue By Region(State)

In [0]:
gold_region = order_items \
    .join(orders, "order_id", "inner") \
    .join(customers, "customer_id", "inner") \
    .filter(col("order_status") == "delivered") \
    .groupBy("customer_state") \
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("price"), 2).alias("total_revenue")
    ) \
    .orderBy(col("total_revenue").desc()) \
    .withColumn("_gold_time", current_timestamp())

gold_region.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.gold.revenue_by_region")

####Monthly order Trends

In [0]:
gold_monthly = orders \
    .filter(col("order_status") == "delivered") \
    .withColumn("order_year", year(col("order_purchase_timestamp"))) \
    .withColumn("order_month", month(col("order_purchase_timestamp"))) \
    .join(order_items, "order_id", "inner")\
    .groupBy("order_year", "order_month") \
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("price"), 2).alias("total_revenue")
    ) \
    .orderBy("order_year", "order_month") \
    .withColumn("_gold_time", current_timestamp())

gold_monthly.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.gold.monthly_trends")

####Payment Method Breakdown

In [0]:
gold_payments = order_payments \
    .join(orders, "order_id", "inner") \
    .filter(col("order_status") == "delivered") \
    .groupBy("payment_type") \
    .agg(
        count("order_id").alias("total_transactions"),
        round(sum("payment_value"), 2).alias("total_value"),
        round(sum("payment_value") / count("order_id"), 2) \
            .alias("avg_transaction_value")
    ) \
    .orderBy(col("total_value").desc()) \
    .withColumn("_gold_time", current_timestamp())

gold_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.gold.payment_breakdown")